# 0m · **서버 결과 폴더 s7 -> mosaic rename**

출력 폴더(`outputs/final/`) 안에 **`s7` 이름으로 저장된 학습/eval 결과 폴더**를
`mosaic` 로 바꾼다. 코드에서 태그를 `acm_s7 -> acm_mosaic` 등으로 바꿨으니,
이미 저장된 결과도 폴더명을 맞춰야 리포트가 찾는다.

예: `train/transfer/acm_s7/seed0` -> `train/transfer/acm_mosaic/seed0`

**순서**: ① 스캔(무엇을 바꿀지) -> ② 충돌 점검 -> ③ `EXECUTE=True` 로 실제 rename -> ④ 검증.
체크포인트 파일 내용은 안 건드린다(폴더/파일 **이름만**). 정책 type/param 은 그대로라 로드에 영향 없음.


In [ ]:
import sys, re, shutil
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

ROOT = cf.OUTPUT_BASE          # ~/lerobot_project/outputs/final
print('스캔 루트:', ROOT)
print('존재:', ROOT.is_dir())

# 이름에 s7 이 들어간 폴더 -> mosaic 로. 경계를 지켜 s7 만 바꾼다(다른 글자 안 건드림).
def new_name(name):
    return re.sub(r's7', 'mosaic', re.sub(r'S7', 'MOSAIC', name))

## 1) 스캔 — s7 이름을 가진 폴더/파일 (dry-run)


In [ ]:
# ── 이름에 s7/S7 이 들어간 파일·폴더 전부 찾기 (깊은 것부터: 자식 먼저 rename) ──
if not ROOT.is_dir():
    print('⚠️ 출력 루트가 없다 - LEROBOT_OUTPUT/경로 확인'); hits = []
else:
    hits = []
    for p in ROOT.rglob('*'):
        if re.search(r's7', p.name, re.IGNORECASE):
            hits.append(p)
    hits.sort(key=lambda p: len(p.parts), reverse=True)   # 깊은 경로부터

print(f's7 이름을 가진 항목: {len(hits)}개\n')
plan = []
for p in hits:
    dst = p.with_name(new_name(p.name))
    kind = 'DIR ' if p.is_dir() else 'FILE'
    conflict = dst.exists()
    plan.append((p, dst, conflict))
    rel = p.relative_to(ROOT)
    print(f"  {kind}  {rel}")
    print(f"        -> {dst.name}" + ("   ⚠️ 대상 이미 존재!" if conflict else ""))
if not hits:
    print('바꿀 것 없음 - 이미 전부 mosaic 이거나 s7 결과가 없다.')

## 2) 충돌 점검 — 대상 이름이 이미 있는가


In [ ]:
# ── 충돌(대상 이름이 이미 있는 경우) 점검 ──
conflicts = [(p, d) for (p, d, c) in plan if c]
if conflicts:
    print('⚠️ 아래는 대상 폴더가 이미 있어 자동 rename 안 함 (수동 병합 필요):')
    for p, d in conflicts:
        print('   ', p.relative_to(ROOT), '->', d.name)
    print('\n대개는 s7 로도, mosaic 로도 각각 결과가 있는 경우다. 어느 쪽을 남길지 정한 뒤')
    print('필요하면 아래 EXECUTE 에서 충돌 항목만 손으로 처리할 것.')
else:
    print('충돌 없음 - 그대로 rename 가능 ✅')

## 3) 실행 — `EXECUTE=True` 로 바꾼 뒤 이 셀 다시 실행


In [ ]:
# ── 실제 rename 실행 ──  (위 계획을 확인했으면 EXECUTE=True 로 바꿔 실행)
EXECUTE = False

if not EXECUTE:
    print('DRY-RUN 상태 (아무것도 안 바꿈). 위 계획이 맞으면 EXECUTE=True 로 바꿔 다시 실행.')
else:
    done, skipped = 0, 0
    for p, dst, conflict in plan:
        if not p.exists():           # 부모가 먼저 바뀌어 경로가 이동한 경우
            continue
        # 부모 경로가 이미 바뀌었을 수 있으니 대상 경로를 현재 위치 기준으로 재계산
        dst = p.with_name(new_name(p.name))
        if dst.exists():
            print('skip (대상 존재):', p.relative_to(ROOT)); skipped += 1; continue
        p.rename(dst)
        print('renamed:', p.name, '->', dst.name); done += 1
    print(f'\n완료: {done}개 rename, {skipped}개 skip')

## 4) 검증


In [ ]:
# ── 검증: s7 이름이 남았는지 다시 스캔 ──
left = [p for p in ROOT.rglob('*') if re.search(r's7', p.name, re.IGNORECASE)]
if left:
    print(f'아직 s7 이름 {len(left)}개 남음:')
    for p in left:
        print('   ', p.relative_to(ROOT))
    print('\n충돌로 skip 된 것들 - 위 충돌 목록 참고해 수동 처리.')
else:
    print('✅ s7 이름 폴더/파일 없음 - 전부 mosaic 으로 통일됨')

# 리포트가 잘 잡는지: acm_mosaic 결과 확인
for task in ('transfer', 'insertion'):
    d = ROOT / 'eval_clean' / task / 'acm_mosaic'
    if d.is_dir():
        print(f'  eval_clean/{task}/acm_mosaic 존재 ✅')